## Enrichment

**Modification Objective:** Make explicit implicit information that is not directly represented in the events but is needed or useful for understanding the domain.

**Motivation:** Relevant domain information may be implicit in the recorded event data rather than represented directly. Deriving and explicitly representing this information can facilitate the interpretation and further analysis of the event log.

**Precondition:** Sufficient information from which the implicit domain information can be derived is available, and the derivation logic is specified.

**Approach:** Identify the information required to derive the implicit domain information, apply the corresponding derivation logic, and represent the resulting information as an additional event- or case-level attribute.

**Output:** An enriched event log in which previously implicit domain information is explicitly represented as an event- or case-level attribute.

In [ ]:
import numpy as np
import pandas as pd
import pm4py

# --- Configuration -----------------------------------------------------------
LOG_PATH = "../../data/Road_Traffic_Fine_Management_Process.xes"

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
TIMESTAMP = "time:timestamp"

In [ ]:
event_log = pm4py.read_xes(LOG_PATH)

display(event_log.head())

### Pattern execution

#### 1. Lift event attribute to case attribute

Derive a case-level attribute from the value of a selected event attribute at the unique event of a case that satisfies a selected condition (e.g., an activity label). If the condition matches no event in a case, the lifted value is NaN; if it matches more than one event, an error is raised.

In [ ]:
def lift_event_value_to_case(event_log: pd.DataFrame,
                              attr: str,
                              condition,                           # callable | str | dict
                              new_col_name: str | None = None) -> pd.DataFrame:
    # --- Input checks (short and strict) ---
    if not {CASE_ID, ACTIVITY, attr}.issubset(event_log.columns):  # ensure required cols exist
        missing = {CASE_ID, ACTIVITY, attr} - set(event_log.columns)
        raise KeyError(f"Missing columns: {missing}")
    if not (callable(condition) or isinstance(condition, (str, dict))):
        raise TypeError("`condition` must be a callable, string (activity) or dict (exact matches).")

    # --- Build a boolean mask for rows that satisfy the condition P ---
    if callable(condition):
        # condition should return a boolean Series aligned to event_log.index
        mask = condition(event_log)
        mask = mask.astype(bool)
        cond_label = getattr(condition, "__name__", "P")
    elif isinstance(condition, str):                                  # string → activity equals that string
        mask = event_log[ACTIVITY] == condition
        cond_label = condition                                      # used for naming the new column
    else:                                                           # dict → all (col == value) must hold
        raise ValueError(f"Unparsable condition: {condition}")

    # --- Decide the output column name (attr@condition) ---
    if new_col_name is None:
        # sanitize for a nicer, safe column name
        new_col_name = "".join(ch if ch.isalnum() or ch in "_@" else "_" for ch in f"{attr}@{cond_label}")

    # --- Keep only candidate events that match P and carry the attr ---
    cand = event_log.loc[mask, [CASE_ID, attr]]                     # [case id, value] pairs for matches

    # --- Count matches per case to detect uniqueness ---
    counts = cand.groupby(CASE_ID).size() if not cand.empty else pd.Series(dtype=int)
    multi_cases = counts[counts > 1].index                           # cases with >1 match (error)

    # --- Raise error if multiple matches are found in any case ---
    if len(multi_cases):
        raise ValueError(f"Condition matched more than one event in cases: {list(multi_cases)}")

    # --- Prepare the result series for all cases (default NaN for 'no match') ---
    cases = event_log[CASE_ID].unique()                              # all cases present in the log
    values = pd.Series(np.nan, index=cases, dtype=object)            # case → lifted value (NaN by default)

    # --- Fill values for unique cases (take that single value) ---
    if not cand.empty:
        single_vals = cand.groupby(CASE_ID)[attr].first()            # exactly one row per matched case
        values.loc[single_vals.index] = single_vals                  # assign lifted values

    # --- Build and return a simple case-table with the new column only ---
    out = pd.DataFrame(index=cases)                                  # one row per case
    out.index.name = CASE_ID
    out[new_col_name] = values                                       # lifted value column
    return out

In [ ]:
lifted = lift_event_value_to_case(event_log, attr="amount", condition="Add penalty")
display(lifted)

#### 2. Weekday enrichment

Derive, for each event, the weekday of its timestamp as an integer in [1..7], assuming the week starts on Monday (Monday=1).

In [ ]:
weekday_log = event_log.copy()

# dt.weekday is 0=Monday...6=Sunday, so dt.weekday + 1 gives the day-of-week index under the assumption that 1=Monday...7=Sunday
weekday_log["weekDay"] = weekday_log[TIMESTAMP].dt.weekday + 1

display(weekday_log[[CASE_ID, ACTIVITY, TIMESTAMP, "weekDay"]].head())

#### 3. Stateful enrichment

Derive, for each event, a result computed from a case-level state that is updated incrementally as events are scanned in case order, e.g., whether a fine was sent within 90 days of being created.

In [ ]:
def update_case_state(state: dict, event) -> dict:
    """
    Given previous state (dict) and current event (Series),
    return a NEW state dict (functional style).
    """
    activity = event[ACTIVITY]
    time = event[TIMESTAMP]

    match activity:
        case "Create Fine":
            state["create_time"] = time

        case "Send Fine":
            state["result"] = ((time - state["create_time"]) <= pd.Timedelta(days=90))

    return state


def scan_case(case_events: pd.DataFrame, update_fn) -> pd.DataFrame:
    state = {}
    states = []

    for _, event in case_events.iterrows():          # <-- Series, so event["concept:name"] works
        state = update_fn(state, event)              # update_fn can use event[ACTIVITY], event[TIMESTAMP]
        states.append(state)

    out = case_events.copy()
    out["state"] = states
    out["result"] = [s.get("result") for s in states]
    return out


stateful_log = event_log.groupby(CASE_ID, group_keys=False).apply(
    lambda g: scan_case(g, update_fn=update_case_state)
)

display(stateful_log)

#### 4. Forward-fill enrichment

Derive, for each event, the value of an attribute that is only recorded at events of a specific activity (e.g., a lab test), by propagating the last recorded value forward to the following events of the same case, in event order, until a new value is recorded.

In [ ]:
SEPSIS_LOG_PATH = "../../data/SepsisCases2020EventLog.xes"

sepsis_log = pm4py.read_xes(SEPSIS_LOG_PATH)

display(sepsis_log.head())

In [ ]:
def forward_fill_attribute_from_activity(event_log: pd.DataFrame,
                                          attr: str,
                                          activity: str) -> pd.DataFrame:
    # --- Input checks ---
    if not {CASE_ID, ACTIVITY, attr}.issubset(event_log.columns):
        missing = {CASE_ID, ACTIVITY, attr} - set(event_log.columns)
        raise KeyError(f"Missing columns: {missing}")

    out = event_log.copy()
    out[attr] = out[attr].where(out[ACTIVITY] == activity)  # keep the value only where it is originally recorded, NaN elsewhere
    out[attr] = out.groupby(CASE_ID)[attr].ffill()           # propagate the last recorded value forward within each case, in event order
    return out


leucocytes_log = forward_fill_attribute_from_activity(sepsis_log, attr="Leucocytes", activity="Leucocytes")

display(leucocytes_log[[CASE_ID, ACTIVITY, TIMESTAMP, "Leucocytes"]].head(30))